<a href="https://colab.research.google.com/github/2022311057/Graduation_research/blob/main/Extracting_parts_of_speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 設定ファイルの場所を修正するコマンド
!mkdir -p /usr/local/etc
!cp /etc/mecabrc /usr/local/etc/mecabrc

# MeCabと辞書をインストールするコマンド
!apt-get -q -y install sudo file mecab libmecab-dev mecab-ipadic-utf8
!pip install mecab-python3

#辞書ライブラリ（ipadic）をインストール
!pip install ipadic

!pip install unidic-lite

Reading package lists...
Building dependency tree...
Reading state information...
libmecab-dev is already the newest version (0.996-14build9).
mecab-ipadic-utf8 is already the newest version (2.7.0-20070801+main-3).
mecab is already the newest version (0.996-14build9).
file is already the newest version (1:5.41-3ubuntu0.1).
sudo is already the newest version (1.9.9-1ubuntu2.5).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
import MeCab
import pandas as pd
import collections
import ipadic
import os

# =================================================================
INPUT_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/study1/source_text.txt"
# =================================================================
OUTPUT_DIR = os.path.dirname(INPUT_FILE_PATH)

# 分析対象の品詞（記号は「区切り」として使うためここには含めない）
FILES = {
    "名詞": "nouns.txt",
    "動詞": "verbs.txt",
    "形容詞": "adjectives.txt",
    "助詞": "particles.txt",
    "助動詞": "auxiliary_verbs.txt",
    "副詞": "adverbs.txt",
    "接続詞": "conjunctions.txt",
    "連体詞": "adnominals.txt",
    "感動詞": "interjections.txt"
}
TARGET_POS_LIST = list(FILES.keys())

# 区切りとみなす記号（これらが来たら「文末」→「文頭」と判断する）
DELIMITERS = ["記号"]

words_by_pos = {k: set() for k in FILES.keys()}

def analyze_text(filepath):
    try:
        tagger = MeCab.Tagger(ipadic.MECAB_ARGS)
    except RuntimeError:
        tagger = MeCab.Tagger()

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
    except FileNotFoundError:
        print(f"エラー: ファイルが見つかりません: {filepath}")
        return None, None, None

    # 3つのカウンタを用意
    start_counts = collections.defaultdict(int)      # 文頭に出た回数
    end_counts = collections.defaultdict(int)        # 文末に出た回数
    transition_counts = collections.defaultdict(lambda: collections.defaultdict(int)) # 遷移回数

    node = tagger.parseToNode(text)

    prev_pos = None
    is_start_of_sentence = True # 最初は文頭扱い

    while node:
        features = node.feature.split(',')
        pos = features[0]

        if pos == "BOS/EOS":
            node = node.next
            continue

        # 単語保存（単語リスト用）
        if len(features) > 6 and features[6] != "*":
            word = features[6]
        else:
            word = node.surface

        if pos in words_by_pos:
            words_by_pos[pos].add(word)

        # --- ロジック部分 ---

        # 1. もし「記号（区切り）」が来た場合
        if pos in DELIMITERS:
            if prev_pos is not None:
                # 直前の品詞は「文末」である
                end_counts[prev_pos] += 1

            # 次に来る品詞は「文頭」になるフラグを立てる
            is_start_of_sentence = True

            # 記号自体は遷移行列の前・後には含めないため、prev_posをリセット
            prev_pos = None

        else:
            # 2. 通常の品詞が来た場合
            current_label = pos if pos in TARGET_POS_LIST else "Others"

            # (A) 文頭フラグが立っていたら「文頭カウント」
            if is_start_of_sentence:
                start_counts[current_label] += 1
                is_start_of_sentence = False # フラグ回収

            # (B) 前に通常の品詞がいれば「遷移カウント」
            if prev_pos is not None:
                transition_counts[prev_pos][current_label] += 1

            # 次のループのために現在地を記憶
            prev_pos = current_label

        node = node.next

    return start_counts, end_counts, transition_counts

def save_probabilities(start_counts, end_counts, transition_counts, output_dir):
    labels = TARGET_POS_LIST + ["Others"]

    # --- 1. 文頭確率 (Start Probabilities) ---
    s_series = pd.Series(0, index=labels)
    for pos, count in start_counts.items():
        if pos in labels: s_series[pos] = count

    # 全体数で割って確率化
    start_prob = s_series / s_series.sum()
    start_prob = start_prob.fillna(0)
    start_prob.to_csv(os.path.join(output_dir, "start_probs.csv"), header=True)
    print("\n--- 文頭確率 (Top 3) ---")
    print(start_prob.sort_values(ascending=False).head(3))

    # --- 2. 文末確率 (End Probabilities) ---
    e_series = pd.Series(0, index=labels)
    for pos, count in end_counts.items():
        if pos in labels: e_series[pos] = count

    # 確率化
    end_prob = e_series / e_series.sum()
    end_prob = end_prob.fillna(0)
    end_prob.to_csv(os.path.join(output_dir, "end_probs.csv"), header=True)
    print("\n--- 文末確率 (Top 3) ---")
    print(end_prob.sort_values(ascending=False).head(3))

    # --- 3. 遷移行列 (Transition Matrix) ---
    df = pd.DataFrame(0, index=labels, columns=labels)
    for prev, next_dict in transition_counts.items():
        for curr, count in next_dict.items():
            if prev in labels and curr in labels:
                df.at[prev, curr] = count

    # 行ごとに正規化
    trans_prob = df.div(df.sum(axis=1), axis=0).fillna(0)
    trans_prob.to_csv(os.path.join(output_dir, "transition_matrix.csv"))
    print("\n--- 遷移行列を保存しました ---")

# --- 実行 ---
print(f"解析開始: {INPUT_FILE_PATH}")
s_counts, e_counts, t_counts = analyze_text(INPUT_FILE_PATH)

if s_counts:
    save_probabilities(s_counts, e_counts, t_counts, OUTPUT_DIR)

    # 単語リスト保存
    for pos, word_set in words_by_pos.items():
        save_name = os.path.join(OUTPUT_DIR, FILES[pos])
        with open(save_name, "w", encoding="utf-8") as f:
            for w in sorted(list(word_set)):
                f.write(w + "\n")
    print("\n全処理完了。")

解析開始: /content/drive/MyDrive/Colab Notebooks/study1/source_text.txt
エラー: ファイルが見つかりません: /content/drive/MyDrive/Colab Notebooks/study1/source_text.txt


In [ ]:
import MeCab
import pandas as pd
import collections
import ipadic
import os

# =================================================================
# 【設定】ログにあったファイルパスをセットしています
INPUT_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/study1/source_text.txt"
# =================================================================

# 出力先を入力ファイルと同じフォルダに設定
OUTPUT_DIR = os.path.dirname(INPUT_FILE_PATH)

# 【改良点】分析対象の品詞を増やしました
# ラップや歌詞の「リズム（記号）」や「つなぎ（接続詞）」を捉えられるようになります
FILES = {
    "名詞": "nouns.txt",
    "動詞": "verbs.txt",
    "形容詞": "adjectives.txt",
    "助詞": "particles.txt",
    "助動詞": "auxiliary_verbs.txt",

    # --- 今回追加した品詞 ---
    "副詞": "adverbs.txt",          # 「既に」「もっと」など
    "接続詞": "conjunctions.txt",    # 「でも」「けど」など
    "連体詞": "adnominals.txt",      # 「あの」「この」「大きな」など
    "記号": "symbols.txt",           # 「、」「。」「…」など（リズムの区切りに重要）
    "感動詞": "interjections.txt"    # 「ああ」「ええ」など（ラップの合いの手に重要）
}

TARGET_POS_LIST = list(FILES.keys())
words_by_pos = {k: set() for k in FILES.keys()}

def analyze_text(filepath):
    # 辞書設定
    try:
        tagger = MeCab.Tagger(ipadic.MECAB_ARGS)
    except RuntimeError:
        tagger = MeCab.Tagger()

    # ファイル読み込み
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
    except FileNotFoundError:
        print(f"エラー: 指定されたパスにファイルが見つかりません。\nパス: {filepath}")
        return None

    # 解析処理
    transition_counts = collections.defaultdict(lambda: collections.defaultdict(int))
    node = tagger.parseToNode(text)
    prev_pos = None

    while node:
        features = node.feature.split(',')
        pos = features[0] # 品詞の1段階目（大分類）を取得

        if pos == "BOS/EOS":
            # 文頭・文末の処理（文脈を切る場合はここでprev_posをNoneにする）
            # 今回は文をまたぐ遷移も見るため、そのまま継続します
            # もし「文末（。）の次は必ずリセットしたい」場合はここを調整します
            node = node.next
            continue

        # 単語の取得（原形があれば原形、なければそのまま）
        if len(features) > 6 and features[6] != "*":
            word = features[6]
        else:
            word = node.surface

        # リストにある品詞なら単語セットに追加
        if pos in words_by_pos:
            # 記号などは1文字でも意味があるため、文字数フィルタは主要3品詞に限定
            if pos in ["名詞", "動詞", "形容詞"] and len(word) < 2:
                # 1文字の漢字（目、手、木など）は残した方がいい場合もあるので
                # 必要に応じてこのif文は外してください
                pass
            else:
                words_by_pos[pos].add(word)

        # 遷移カウント用ラベル決定
        current_pos_label = pos if pos in TARGET_POS_LIST else "Others"

        if prev_pos is not None:
            transition_counts[prev_pos][current_pos_label] += 1

        prev_pos = current_pos_label
        node = node.next

    return transition_counts

def save_transition_matrix(transition_counts, output_dir, filename="transition_matrix.csv"):
    # 行列のラベル順序を定義
    labels = TARGET_POS_LIST + ["Others"]
    df = pd.DataFrame(0, index=labels, columns=labels)

    for prev, next_dict in transition_counts.items():
        for curr, count in next_dict.items():
            if prev in labels and curr in labels:
                df.at[prev, curr] = count

    # 確率行列の計算
    prob_df = df.div(df.sum(axis=1), axis=0).fillna(0)

    # 保存
    save_path = os.path.join(output_dir, filename)
    prob_df.to_csv(save_path, encoding="utf-8-sig")

    print("\n--- 遷移確率行列（一部抜粋） ---")
    print(prob_df.round(3))
    print(f"\n遷移行列を保存しました: {save_path}")

# --- 実行部分 ---
print(f"解析対象: {INPUT_FILE_PATH}")

counts = analyze_text(INPUT_FILE_PATH)

if counts:
    # 単語リスト保存
    for pos, word_set in words_by_pos.items():
        save_name = os.path.join(OUTPUT_DIR, FILES[pos])
        with open(save_name, "w", encoding="utf-8") as f:
            sorted_words = sorted(list(word_set))
            for w in sorted_words:
                f.write(w + "\n")
        print(f"[{pos}] {len(word_set)} 語 を {FILES[pos]} に保存しました。")

    # 遷移行列保存
    save_transition_matrix(counts, output_dir=OUTPUT_DIR)

    print("\n完了しました。")

解析対象: /content/drive/MyDrive/Colab Notebooks/study1/source_text.txt
エラー: 指定されたパスにファイルが見つかりません。
パス: /content/drive/MyDrive/Colab Notebooks/study1/source_text.txt


In [ ]:
import MeCab

# 設定：保存するファイル名
FILES = {
    "名詞": "nouns.txt",
    "動詞": "verbs.txt",
    "形容詞": "adjectives.txt",
    "助詞": "particles.txt",
    "助動詞": "auxiliary_verbs.txt"
}

# データの入れ物
words_by_pos = {k: set() for k in FILES.keys()}

def extract_words(filename):
    tagger = MeCab.Tagger()

    try:
        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()
    except FileNotFoundError:
        print(f"エラー: {filename} が見つかりません。")
        return

    node = tagger.parseToNode(text)

    while node:
        features = node.feature.split(',')
        # features[0] は品詞の大分類（名詞, 動詞, etc）
        pos = features[0]

        # features[6] は原形（辞書形）。もし取得できなければ表層形（そのまま）を使う
        # 例: "走っ" -> "走る" に変換して保存したい
        if len(features) > 6 and features[6] != "*":
            word = features[6]
        else:
            word = node.surface

        # ターゲットの品詞ならセットに追加（重複排除のためset使用）
        if pos in words_by_pos:
            # 短すぎる単語や記号を除去するフィルタ（お好みで調整）
            if pos in ["名詞", "動詞", "形容詞"] and len(word) < 2:
                pass # 1文字の単語（"目"や"手"など）を除外したければここで制御
            else:
                words_by_pos[pos].add(word)

        node = node.next

# --- 実行部分 ---
# Colab Notebooks と study1 の間に / を入れます
input_file = "/content/drive/MyDrive/Colab Notebooks/study1/japanese_rap/source_text.txt"  # ここに歌詞や小説のテキストを入れる
print(f"{input_file} から単語を抽出中...")

extract_words(input_file)

# ファイルへの書き出し
for pos, word_set in words_by_pos.items():
    filename = FILES[pos]
    with open(filename, "w", encoding="utf-8") as f:
        # ソートして書き出し
        sorted_words = sorted(list(word_set))
        for w in sorted_words:
            f.write(w + "\n")
    print(f"[{pos}] {len(word_set)} 語 を {filename} に保存しました。")

print("\n完了しました。")

/content/drive/MyDrive/Colab Notebooks/study1/japanese_rap/source_text.txt から単語を抽出中...
[名詞] 98 語 を nouns.txt に保存しました。
[動詞] 44 語 を verbs.txt に保存しました。
[形容詞] 5 語 を adjectives.txt に保存しました。
[助詞] 22 語 を particles.txt に保存しました。
[助動詞] 8 語 を auxiliary_verbs.txt に保存しました。

完了しました。
